# IVF Insight — Phase 4: Predictive Modelling & Prospective Evaluation

**Author:** Senior Biomedical Data Scientist & Research Engineer  
**Source Dataset:** `data/raw/Dataset Clean.xlsx` (5,000 authoritative patient treatment cycles)  
**Validation Strategy:** Prospective Chronological Partition (Cutoff: `2024-01-01`)  

> [!IMPORTANT]
> **Clinical Non-Claim Notice:** Assisted reproductive technology (ART) involves complex sequential clinical decisions. This predictive modeling study is an educational portfolio project. It does **not** constitute a clinical decision support tool, medical device, or basis for clinical management. Predictive importance must **never** be interpreted as causal biological efficacy.

## 1. Environment Setup & Pipeline Imports

Initialize dependencies and import modular components from `ivf_insight.experiments`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import Image, display

from ivf_insight.eda import load_clean_cohorts
from ivf_insight.governance import ChronologicalValidationManager
from ivf_insight.experiments import (
    SCENARIO_FEATURES,
    run_predictive_experiments,
)

REPORTS_DIR = Path('../reports')
FIGURES_DIR = REPORTS_DIR / 'model_figures'
CM_DIR = REPORTS_DIR / 'confusion_matrices'
print('Loaded experiments module. Reports directory:', REPORTS_DIR.resolve())


## 2. Experimental Scenarios & Cohort Partitions

We evaluate three sequential clinical stages:
1. **Scenario A (Baseline, 4 confirmed features):** Female Age, Male Age, AMH, Number of attempts.
2. **Scenario B (Post-Retrieval, 8 features):** Baseline + Oocyte Maturity (MII, MI, others) + Number of fertilized eggs.
3. **Scenario C (Pre-Transfer, 14 features):** Post-retrieval + Morphological Grades 1-4 + Total Embryos + Surplus Cryopreservation (`frezzing`).

Evaluated across two non-interchangeable analytical cohorts:
- **Full-Cycle Cohort (N = 5,000):** Historical Train = 3,542 cycles, Prospective Test = 1,458 cycles.
- **Embryo-Transfer Cohort (N = 3,750):** Historical Train = 2,719 cycles, Prospective Test = 1,031 cycles.

In [ ]:
for s_name, feats in SCENARIO_FEATURES.items():
    print(f'{s_name} ({len(feats)} features): {feats}')


## 3. Prospective Evaluation Metrics & Results Summary

Load the audited metrics evaluated strictly on the 2024–2025 prospective holdout with 95% bootstrap confidence intervals.

In [ ]:
metrics_df = pd.read_csv(REPORTS_DIR / 'model_metrics.csv')
display_cols = ['cohort', 'scenario', 'model', 'roc_auc', 'roc_auc_ci_95', 'pr_auc', 'balanced_accuracy', 'accuracy', 'f1_score', 'brier_score']

print('=== Full-Cycle Cohort Results ===')
display(metrics_df[metrics_df['cohort'] == 'Full-Cycle Cohort'][display_cols])

print('=== Embryo-Transfer Cohort Results ===')
display(metrics_df[metrics_df['cohort'] == 'Embryo-Transfer Cohort'][display_cols])


## 4. Incremental Prognostic Gain Across Milestones

Observe the sequential gain in discrimination as laboratory and embryological data become available.

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'model_comparison_barchart.png')))


## 5. ROC & Precision-Recall Curves (Prospective Holdout)

Receiver Operating Characteristic (ROC) and Precision-Recall (PR) curves for the Embryo-Transfer Cohort.

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'roc_curves_transfer_cohort.png')))
display(Image(filename=str(FIGURES_DIR / 'pr_curves_transfer_cohort.png')))


## 6. Calibration & Reliability Assessment

Assess agreement between predicted probabilities and observed empirical positive rates.

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'calibration_curves_transfer_cohort.png')))


## 7. Confusion Matrices

Examine confusion matrices for Scenario C Pre-Transfer models.

In [ ]:
display(Image(filename=str(CM_DIR / 'embryo-transfer_cohort_scenario_c_random_forest_cm.png')))
display(Image(filename=str(CM_DIR / 'embryo-transfer_cohort_scenario_c_logistic_regression_cm.png')))


## 8. Model Explainability: Feature Attribution & Odds Ratios

Investigate Logistic Regression standardized odds ratios and Random Forest Gini importance for Scenario C.

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'feature_importance_scenario_c.png')))


## 9. Critical Biomedical Findings & Research Governance Summary

1. **The Conflation Illusion:** Full-Cycle Cohort achieves ROC-AUC = 0.903 in Scenario C because `Number of embryos == 0` identifies non-transfer cycle cancellations. In the true Embryo-Transfer Cohort, performance reaches a realistic ROC-AUC = 0.729.
2. **Top Prognostic Signals:** Surplus embryo vitrification (`frezzing`, OR = 2.39, Gini = 0.381) and normal fertilization yield (`Number of fertilized eggs`) provide the strongest out-of-sample discrimination.
3. **Prospective Practice Drift:** Positive rate shifted from 61.8% to 78.7% in the transfer cohort, reinforcing the necessity of chronological validation over random K-fold splits.

**No clinical deployment claims are made. Unresolved target definitions are documented in `reports/research_questions.md`.**